In [1]:
import numpy as np 
import pandas as pd 
import tensorflow as tf
from keras.models import Sequential
from keras.models import load_model
from keras.layers import Dense
from keras.optimizers import Adam
from keras.layers import Dense, Dropout
import math
import datasets

2025-03-28 15:27:15.504050: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-28 15:27:17.586622: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/ishan/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Solving the formula



In [ ]:
# Convert numbers written as words to numbers 

from word2number import w2n  

def convert_number_words(text):  
    words = text.split()  # Tokenize by spaces  
    converted_words = []  

    for word in words:  
        try:  
            converted_words.append(str(w2n.word_to_num(word)))  # Convert if it's a number  
        except ValueError:  
            converted_words.append(word)  # Keep original if it's not a number  

    return " ".join(converted_words)  

In [4]:
# A cleaning fucntion which removes question marks, fullstops and commas

import re

def cleaner(s):
    return re.sub(r"[?,.]","",s)

In [5]:
# Takes questions as input and replaces the numbers with variables 

def form2gen(st):
    st = cleaner(st)
    st = convert_number_words(st)
    f=''
    c=0
    values={}
    tok=st.split(" ")
    for i in range(len(tok)):
        if tok[i].isnumeric():
            if c<10:
                f+='n_0'+str(c)
                values['n_0'+str(c)] = tok[i]
            else:
                f+='n_'+str(c)
                values['n_'+str(c)] = tok[i]
            c+=1
            
        else:
            f+=tok[i]
        f+=" "
    return(f,values)

In [6]:
# Training dataset

df=pd.read_csv("/home/ishan/Downloads/mawps_train.csv")
total_problem_phrases = list(df['Question'])
total_formula_phrases = list(df['Equation'])
df.head()

,Question,Equation,Answer,Numbers
0,Mary is baking a cake . The recipe wants N_00 ...,N_00 - N_01,6.0,8.0 2.0
1,There are N_00 erasers and N_01 scissors in th...,N_00 + N_02,270.0,139.0 118.0 131.0
2,One pencil weighs N_00 grams . How much do N_0...,N_00 * N_01,141.5,28.3 5.0
3,Zoe was unboxing some of her old winter clothe...,N_00 * ( N_01 + N_02 ),80.0,8.0 4.0 6.0
4,"Keith grew N_00 cantelopes , Fred grew N_01 ca...",N_00 + N_01 + N_02,65.0,29.0 16.0 20.0


In [7]:
# Test dataset

df=pd.read_csv("/home/ishan/Downloads/sigmadolphin.csv")
test_total_problem_phrases = list(df['Problem'])
test_answers = list(df['Answer'])
df.head()

,Number Word Version,Problem,Answer
0,number_word_std.test,What is the sum of 1 and 1?,2.0
1,number_word_std.test,When the smaller of two consecutive integers i...,49.0
2,number_word_std.test,The product of two positive consecutive odd in...,40.0
3,number_word_std.test,The sum of three consecutive positive integers...,83.0
4,number_word_std.test,The sum of 3 consecutive integers is 54. What ...,19.0


In [ ]:
# Main model which includes Encoder Decoder with attention

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Attention
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

problem_phrases = total_problem_phrases
formula_phrases = [f'<start> {phrase} <end>' for phrase in total_formula_phrases]

def tokenize_and_pad(sequences, max_len):
    tokenizer = Tokenizer(filters='')
    tokenizer.fit_on_texts(sequences)
    sequences = tokenizer.texts_to_sequences(sequences)
    padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')
    return padded_sequences, tokenizer

max_len_eng = max(len(seq.split()) for seq in problem_phrases)
max_len_fr = max(len(seq.split()) for seq in formula_phrases)

input_sequences, input_tokenizer = tokenize_and_pad(problem_phrases, max_len_eng)
output_sequences, output_tokenizer = tokenize_and_pad(formula_phrases, max_len_fr)

num_encoder_tokens = len(input_tokenizer.word_index) + 1
num_decoder_tokens = len(output_tokenizer.word_index) + 1

encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(num_encoder_tokens, 256)(encoder_inputs)
encoder_lstm = LSTM(256, return_sequences=True, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(num_decoder_tokens, 256)(decoder_inputs)
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

attention = Attention()([decoder_outputs, encoder_outputs])
decoder_combined_context = tf.keras.layers.Concatenate(axis=-1)([decoder_outputs, attention])
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_combined_context)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='categorical_crossentropy')

def one_hot_encode(sequences, num_classes):
    return np.array([tf.keras.utils.to_categorical(seq, num_classes=num_classes) for seq in sequences])

decoder_target_data = one_hot_encode(output_sequences[:, 1:], num_decoder_tokens)

model.fit(
    [input_sequences, output_sequences[:, :-1]], 
    decoder_target_data,
    batch_size=64,
    epochs=50,                                    # Adjust the number of epochs to achieve better results 
    validation_split=0.1
)

encoder_model = Model(encoder_inputs, [encoder_outputs] + encoder_states)

decoder_state_input_h = Input(shape=(256,))
decoder_state_input_c = Input(shape=(256,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_hidden_state_input = Input(shape=(max_len_eng, 256))
decoder_outputs, state_h, state_c = decoder_lstm(decoder_embedding, initial_state=decoder_states_inputs)

attention_inf = Attention()([decoder_outputs, decoder_hidden_state_input])
decoder_combined_context_inf = tf.keras.layers.Concatenate(axis=-1)([decoder_outputs, attention_inf])
decoder_outputs = decoder_dense(decoder_combined_context_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs + [decoder_hidden_state_input],
    [decoder_outputs] + [state_h, state_c]
)

def decode_sequence(input_seq):
    encoder_outputs, state_h, state_c = encoder_model.predict(input_seq)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = output_tokenizer.word_index['<start>']
    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict(
            [target_seq, state_h, state_c, encoder_outputs]
        )
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = output_tokenizer.index_word[sampled_token_index]
        decoded_sentence += ' ' + sampled_word
        if sampled_word == '<end>' or len(decoded_sentence) > max_len_fr:
            stop_condition = True
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        state_h, state_c = h, c
    return decoded_sentence.replace('<start>', '').replace('<end>', '').strip()

In [ ]:
# Test over the test data

import numpy as np
import random
corrects=[]
countA=0
answers=[]
for r in range(len(test_total_problem_phrases)):
    #r = random.randint(0,100)
    # Select a sample problem phrase
    cleaned = form2gen(test_total_problem_phrases[r])
    sample_problem = cleaner(cleaned[0])  # Change index to test different inputs
    vals = cleaned[1]
    sample_output = test_answers[r]
    
    # Tokenize and pad the input sequence
    input_seq = input_tokenizer.texts_to_sequences([sample_problem])
    input_seq = pad_sequences(input_seq, maxlen=max_len_eng, padding='post')

    # Decode the sequence using the trained model
    output_formula = decode_sequence(input_seq)
    #if checker(sample_output,output_formula):
    if solver(output_formula,vals)==sample_output:
        countA+=1
        corrects+=[[sample_output,output_formula]]
    answers+=[sample_problem,output_formula]
# Print the predicted formula
#print(f"Predicted Formula: {output_formula}")

In [ ]:
# Final Score

print(countA,"/",len(test_answers))